# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarveyWebbs/ML-Basics/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*Unit of Analysis: One row represents exactly one unique piece of content (content_hash_id).
Time Window: We are evaluating a single, mid-panel month: March 2026 (March 1 to March 31, 2026). The final month (June 2026) is strictly excluded to serve as our unseen test set later..*

In [2]:
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.execute(f"""
    CREATE OR REPLACE VIEW march_performance AS
    SELECT
        content_hash_id,
        MIN(report_date) as first_date,
        MAX(report_date) as last_date,
        SUM(gsc_impressions) as total_gsc_impressions,
        SUM(gsc_clicks) as total_gsc_clicks,
        (SUM(gsc_impressions) > 0) AS has_visibility
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
    GROUP BY 1
""")
print("Base March 2026 view created successfully.")


Base March 2026 view created successfully.


## 2. Fields: feature / label / context / excluded

*Features (The 5 inputs):
content_type: Knowable at the decision moment because the CMS category is permanently assigned.
content_age_days: Knowable at the decision moment because publication timestamp is historical.
days_since_updated: Knowable at the decision moment because we calculate the gap from the last edit to the start of March.
publish_day_of_week: Knowable at the decision moment because the calendar day is fixed.
client_hash_id: Knowable at the decision moment because domain ownership is static.
Label (Target): total_gsc_clicks (The total clicks accumulated by the end of March 2026).
Context: has_visibility (A boolean flag used to filter out dead content).
Excluded: total_gsc_impressions. Why: This is a data leak. We cannot predict March clicks using March impressions, because we wouldn't know the total impressions on March 1st!*

In [3]:
from sklearn.linear_model import LinearRegression
import numpy as np

query_features = f"""
SELECT
    c.content_hash_id,
    c.content_type,
    DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') AS content_age_days,
    DATE_DIFF('day', c.content_updated_date, DATE '2026-03-01') AS days_since_updated,
    EXTRACT(DOW FROM c.content_created_date) AS publish_day_of_week,
    c.client_hash_id,
    p.total_gsc_impressions,
    p.total_gsc_clicks
FROM read_parquet('{rel}/dim_content.parquet') c
JOIN march_performance p ON c.content_hash_id = p.content_hash_id
WHERE content_age_days >= 0
"""
feature_frame = con.sql(query_features).df()

ml_df = feature_frame.dropna(subset=['content_age_days', 'days_since_updated', 'total_gsc_impressions', 'total_gsc_clicks']).copy()
y = np.log1p(ml_df['total_gsc_clicks'])

print("--- THE TRAP ---")
X_leaked = ml_df[['content_age_days', 'days_since_updated', 'total_gsc_impressions']]
score_leaked = LinearRegression().fit(X_leaked, y).score(X_leaked, y)
print(f"R-squared with LEAKED feature (Impressions): {score_leaked:.4f} <- Fake near-perfect score!")

X_honest = ml_df[['content_age_days', 'days_since_updated']]
score_honest = LinearRegression().fit(X_honest, y).score(X_honest, y)
print(f"R-squared with HONEST features: {score_honest:.4f} <- The real baseline we must beat.")

del feature_frame['total_gsc_impressions']
print("\nExcluded/Dropped 'total_gsc_impressions' from feature_frame.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- THE TRAP ---
R-squared with LEAKED feature (Impressions): 0.3821 <- Fake near-perfect score!
R-squared with HONEST features: 0.0313 <- The real baseline we must beat.

Excluded/Dropped 'total_gsc_impressions' from feature_frame.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Here I am running three queries to prove the assumptions from Section 1: verifying the grain (1-to-1 relationship), confirming the actual dates pulled from the partition, and filtering the active rows using the context boolean*

In [4]:
print("--- QUERY VERIFICATIONS ---")

grain_check = con.sql("SELECT COUNT(*) as total_rows, COUNT(DISTINCT content_hash_id) as unique_ids FROM march_performance").df()
print("1. Grain Check:\n", grain_check.to_string(index=False), "\n")

span_check = con.sql("SELECT MIN(first_date) as start_date, MAX(last_date) as end_date FROM march_performance").df()
print("2. Window Span:\n", span_check.to_string(index=False), "\n")

avail_check = con.sql("SELECT COUNT(*) as surviving_active_rows FROM march_performance WHERE has_visibility IS TRUE").df()
print("3. Availability:\n", avail_check.to_string(index=False))


--- QUERY VERIFICATIONS ---
1. Grain Check:
  total_rows  unique_ids
     331437      331437 



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2. Window Span:
 start_date   end_date
2026-03-01 2026-03-31 

3. Availability:
  surviving_active_rows
                176738


## 4. Data limits

*What this data can never tell us:
Unbalanced History: As verified in dim_clients, different clients installed tracking code (gsc_data_start / ga4_data_start) at totally different times. We cannot compare raw historical lifetime totals between two domains, because one might have 4 years of data and another might only have 6 months.
Causality: This is an observational dataset. Even if we find a strong correlation (e.g., updating content heavily correlates with more traffic), we cannot mathematically prove the update caused the traffic increase.
Platform Source-of-Truth: Early rows might be GSC-only (Google Search Console) before GA4 (Google Analytics 4) was fully implemented. Therefore, engagement metrics (like bounce rate or sessions) might be totally null for older content in this dataset.*

In [5]:
query_limits = f"""
    SELECT
        MIN(gsc_data_start) as oldest_client_start,
        MAX(gsc_data_start) as newest_client_start
    FROM read_parquet('{rel}/dim_clients.parquet')
"""
limits_df = con.sql(query_limits).df()
print("Proof of Unbalanced History:")
display(limits_df)


Proof of Unbalanced History:


,oldest_client_start,newest_client_start
0,2025-01-27,2026-06-02


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.